# Assignment 4

In [1]:
import pandas as pd 

## Q1: Build Your Personalized Knowledge Base

Take your college roll number and extract its digits. Build a pandas DataFrame with exactly **6 FAQ entries**:

- **4 fixed entries** (given in the table below)
- **2 entries constructed from your own roll number digits**, as follows:

### Steps to construct the 2 custom entries
1. Take the **last two digits** of your roll number.
2. For each digit `d`, compute:
```python
   category = ["billing", "account", "general"][d % 3]
```
3. Invent one realistic **question + answer + 3 keywords** per entry that fits the assigned category.
   - *Example:* if `d % 3` gives `"account"`, write a question like *"How do I update my registered mobile number?"*

### Example
```python
# Example roll number ...23 -> digits 2, 3
# digit 2 -> category[2 % 3] = "general"
# digit 3 -> category[3 % 3] = "billing"
```

In [2]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or netbanking.", 
     "keywords": "pay payment upi fee", "category": "billing"}
]
custom_entries = [
    {"question": "why was my late payment fee charged",
     "answer": "A late fee is added if payment is made after the due date shown on your invoice.",
     "keywords": "late fee penalty", "category": "billing"},
    {"question": "how do i update my registered mobile number",
     "answer": "Go to Account Settings > Profile > Edit Mobile Number, then verify with the OTP sent to your new number.",
     "keywords": "mobile number update", "category": "account"}
]
faq_entries = fixed_entries + custom_entries
df = pd.DataFrame(faq_entries)
print(df)

                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4          why was my late payment fee charged   
5  how do i update my registered mobile number   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3          You can pay via UPI, card, or netbanking.     pay payment upi fee   
4  A late fee is added if payment is made after t...        late fee penalty   
5  Go to Account Settings > Profile > Edit Mobile...    mobile number update   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  billing  
5  account

## Q2 : Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence

In [3]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for _,row in df.iterrows():
        entry_words = set(row["question"].lower().split()) | set(row["keywords"].lower().split())
        matches = query_words & entry_words
        score = len(matches)

        if score > 0:
            confidence = score / len(query_words)
            results.append({
                "question" : row["question"],
                "answer" : row["answer"],
                "category" : row["category"],
                "confidence": round(confidence, 2)
            })

    results.sort(key=lambda x: x["confidence"], reverse=True)
    return pd.DataFrame(results)

score_query("how do I pay my fee", df)

,question,answer,category,confidence
0,how can i pay the fee,"You can pay via UPI, card, or netbanking.",billing,0.67
1,how do i update my registered mobile number,Go to Account Settings > Profile > Edit Mobile...,account,0.67
2,why was my late payment fee charged,A late fee is added if payment is made after t...,billing,0.33
3,what is the annual fee,The annual fee is Rs 500.,billing,0.17
4,how to reset password,Go to Settings > Reset Password.,account,0.17


## Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [4]:
def same_category(category_name, df):
    matches = df[df["category"] == category_name]
    return matches["question"]
print(same_category("account", df))

1                          how to reset password
5    how do i update my registered mobile number
Name: question, dtype: str


## Q4 : Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.

In [6]:
entry_index = 0;
new_keyword = input("Enter a new keyword to add : ")
df.at[entry_index, "keywords"] += new_keyword
df.to_csv("1024170164_faq_data.csv", index=False)
print(f"Updated entry {entry_index} keywords: {df.at[entry_index, 'keywords']}")
print("Saved to 1024170164_faq_data.csv")

Enter a new keyword to add :  money


Updated entry 0 keywords: fee cost price chargemoneymoney
Saved to 1024170164_faq_data.csv


## Q5 : Using groupby, print how many FAQ entries you have per category.

In [7]:
print(df.groupby("category").size())

category
account    2
billing    3
general    1
dtype: int64


## Q6 : Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [8]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []
    for _, row in df.iterrows():
        entry_words = set(row["question"].lower().split()) | set(row["keywords"].lower().split())
        score = len(query_words & entry_words)
        if score > 0:
            confidence = score / len(query_words)
            results.append({
                "question": row["question"],
                "confidence": round(confidence, 2)
            })
    if not results:
        print("No match found.")
        return
    # find the highest confidence
    best_score = max(r["confidence"] for r in results)
    # keep only entries with that highest score
    best_matches = [r for r in results if r["confidence"] == best_score]

    if len(best_matches) > 1:
        print("Tie! Multiple entries match equally well:")
    else:
        print("Best match:")

    for r in best_matches:
        print(r)


# Query that ties (matches both "fee" entries)
print("Query: 'fee'")
score_query("fee", df)
print()
# Query that doesn't tie
print("Query: 'how do I pay my fee'")
score_query("how do I pay my fee", df)

Query: 'fee'
Tie! Multiple entries match equally well:
{'question': 'what is the annual fee', 'confidence': 1.0}
{'question': 'how can i pay the fee', 'confidence': 1.0}
{'question': 'why was my late payment fee charged', 'confidence': 1.0}

Query: 'how do I pay my fee'
Tie! Multiple entries match equally well:
{'question': 'how can i pay the fee', 'confidence': 0.67}
{'question': 'how do i update my registered mobile number', 'confidence': 0.67}
